In [1]:
import uuid
from dotenv import load_dotenv
from modules import logging
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_openai import ChatOpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.schema import BaseMessage
from typing_extensions import Annotated, Literal, Sequence, TypedDict
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display
import pprint
from langchain_core.runnables import RunnableConfig
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain.globals import set_verbose, set_debug

ImportError: cannot import name 'logging' from 'modules' (unknown location)

In [ ]:
# LangSmith Logging
logging.langsmith("Model_RAG")
# API 키 정보 로드
load_dotenv()

In [3]:
session_id = str(uuid.uuid4())
memory = MemorySaver()

In [ ]:
set_debug(True)
set_verbose(True)

In [ ]:
print(session_id)

In [ ]:
ollama_embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

loaded_db = FAISS.load_local(
    folder_path="faiss_db",
    index_name="meritz_index",
    embeddings=ollama_embeddings,
    allow_dangerous_deserialization=True,
)

In [6]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [7]:
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=loaded_db.as_retriever(),
    llm=llm
)

In [8]:
class state(TypedDict):
    # 메시지 관리 (사용자와 시스템의 대화 기록)
    messages: Annotated[Sequence[BaseMessage], "add_messages"]
    # 세션 ID (각 워크플로우 인스턴스에 고유한 ID)
    session_id: str
    # 기타 메타데이터 (사용자 정보, 소스 등)
    metadata: dict
    # 검색된 문서 리스트
    retrieved_docs: dict  # 검색 결과 (MultiQueryRetriever의 반환 값)
    # 문서 등급
    document_grading: list
    # 보험 등급
    insurance_grading : str
    # 재 생성 된 답변
    rewritten_question : list
    # 생성된 최종 답변
    generated_response: str  # 시스템의 최종 답변

In [9]:
# 답변 분류


def Router(state: state) -> state:
    """
    Determines whether the question is related to car insurance.

    Args:
        Status (message): current status

    Return:
        str: a decision on the relevance of questions to car insurance
    """

    print("---CHECK Rout---")

    # Data model
    class grade(BaseModel):
        """Binary score for relevance check."""

        binary_score: str = Field(description="Relevance score 'Relevant' or 'Not Relevant'")

    # LLM
    model = ChatOpenAI(temperature=0, model="gpt-4o-mini", streaming=True)

    # LLM with tool and validation
    llm_with_tool = model.with_structured_output(grade)

    # Prompt
    Router_prompt = PromptTemplate(
        template="""
        
        The Router must respond appropriately to the user's questions based on the modified status {state}. Follow these steps:
        Analyze the user's question to determine its relevance to auto insurance. Follow the structured approach provided and respond accurately.

        # Steps

        1. **Analyze Question**: Evaluate the user's question to identify if it pertains to auto insurance topics.
        2. **Determine Relevance**: 
        - If the question is about auto insurance, classify it as "relevant."
        - If the question is unrelated to auto insurance, classify it as "not relevant."

        # Notes

        - Make sure to precisely identify whether the question is related to car insurance, as accurate classification is crucial.
        - Avoid adding any extraneous information beyond the relevance classification.

        # Output Format

        - Provide the output as a single word for classification:
        - "Relevant"
        - "Not Relevant"

        # Examples

        - Example 1:
        - **Input**: "What does my auto insurance cover if I'm in an accident?"
        - **Reasoning**: The question clearly pertains to auto insurance coverage.
        - **Output**: "Relevant"

        - Example 2:
        - **Input**: "What are the best vacation spots this summer?"
        - **Reasoning**: The question is about travel, not about auto insurance.
        - **Output**: "Not Relevant"
        
        #Question:  
        {question}
        {state}
        
        #Answer:
        
        """,
        input_variables=["question","state"],
    )
    
    
    # Chain
    chain = Router_prompt | llm_with_tool
    
    messages = state["messages"]
    rewritten_question = state["rewritten_question"]

    if state.get("rewritten_question") is []:
        question = messages[0]
    else:
        question = state["rewritten_question"][-1]["content"]
        
        
    scored_result = chain.invoke({"question": question, "state" : state})

    score = scored_result.binary_score

    
    state["insurance_grading"] = score
    print(f"Document insurance_grading: {state['insurance_grading']}")
    print(f"State: {state}")
    print(f"Messages: {messages}")
    return state

In [10]:
def agent(state: state) -> state:
    if state["insurance_grading"] == "Not Relevant":
        response = "대답할 수 없는 질문입니다."
     
    
    return print(response)

In [11]:
def retriever_tool(state:state) -> state:
    """
    It's a multi-query retriever tool
    Retrieves documents using MultiQueryRetriever based on the current question.
    """
    print("---CALL RETRIEVE TOOL---")
    
    # 사용자의 마지막 질문 또는 rewritten_question이 있는 경우 해당 값을 사용
    # user_question = state["messages"][-1]["content"]
    messages = state["messages"]
    user_question = messages[-1]["content"]
    
    # MultiQueryRetriever에 사용할 프롬프트
    multi_query_prompt = f"""
    You are an expert assistant trained to optimize search queries for retrieving the most relevant documents.
    Given a user question, your task is to generate multiple rephrased versions of the same question. These
    rephrased questions should:
    
    1. Cover different ways a user might phrase the same question.
    2. Maintain the original intent and meaning of the question.
    3. Use synonyms, structural variations, and alternative expressions where applicable.

    For the following user question, generate at least **5 rephrased versions**:
    
    Original Question:
    "{user_question}"
    
    Ensure your rephrased versions are concise, clear, and ready to be used in a search query. Output the rephrased questions as a list.
    """
    
    # MultiQueryRetriever로 문서 검색
    retrieved_docs = multi_query_retriever.get_relevant_documents(multi_query_prompt)
    
    # 검색된 문서를 상태에 저장
    state["retrieved_docs"] = retrieved_docs
    return state

In [12]:
# 답변 분류


def grade_documents(state: state) -> state:
    """
    Determines whether the retrieved documents are relevant to the question.

    Args:
        state (messages): The current state

    Returns:
        str: A decision for whether the documents are relevant or not
    """

    print("---CHECK RELEVANCE---")

    # Data model
    class grade(BaseModel):
        """Binary score for relevance check."""

        binary_score: str = Field(description="Relevance score 'Relevant' or 'Not Relevant'")

    # LLM
    model = ChatOpenAI(temperature=0, model="gpt-4o-mini", streaming=True)

    # LLM with tool and validation
    llm_with_tool = model.with_structured_output(grade)

    # Prompt
    prompt = PromptTemplate(
        template="""You are a grader assessing relevance of a retrieved document to a user question. \n 
        Here is the retrieved document: \n\n {context} \n\n
        Here is the user question: {question} \n
        If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant. \n
        Give a binary score 'Relevant' or 'Not Relevant' score to indicate whether the document is relevant to the question.""",
        input_variables=["context", "question"],
    )
    
    
    # Chain
    chain = prompt | llm_with_tool

    messages = state["messages"]
    docs = state["retrieved_docs"]
    
    question = messages[0]["content"]
    context = docs

    scored_result = chain.invoke({"question": question, "context": docs})

    score = scored_result.binary_score

    state["document_grading"] = score
    print(f"Document Grading: {state['document_grading']}")
    return state

In [13]:
# 조건부 연결 추가
def conditional_decision(state: state) -> state:
    """
    조건부 경로를 결정하는 함수.
    'grade_documents' 결과에 따라 경로가 나뉨.
    """
    if state.get("document_grading") == "Relevant":
        return "generate"
    
    if state.get("document_grading") == "Not Relevant":
        return "rewrite"

In [14]:
# 조건부 연결 추가
def conditional_retriever(state: state) -> state:
    """
    조건부 경로를 결정하는 함수.
    'insurance_grading' 결과에 따라 경로가 나뉨.
    """
    if state.get("insurance_grading") == "Relevant":
        return "retriever_tool"
    
    if state.get("insurance_grading") == "Not Relevant":
        return "agent"

In [15]:
# Rewrite 노드

def rewrite(state):
    """
    Transform the query to produce a better question.

    Args:
        state (messages): The current state

    Returns:
        dict: The updated state with re-phrased question
    """

    print("---TRANSFORM QUERY---")
    messages = state["messages"]
    question = messages[0].content

    msg = [
        HumanMessage(
            content=f""" \n 
    Look at the input and try to reason about the underlying semantic intent / meaning. \n 
    Here is the initial question:
    \n ------- \n
    {question} 
    \n ------- \n
    Formulate an improved question: """,
        )
    ]

    # Grader
    model = ChatOpenAI(temperature=0, model="gpt-4o-mini", streaming=True)
    response = model.invoke(msg)
    state["rewritten_question"] = response
    
    
    return state

In [16]:
def generate(state: state) -> state:
    """
    The node that generates the answer
    Generates the final response based on the user's question and relevant documents.
    """
    print("---GENERATE RESPONSE NODE ACTIVATED---")
    
    user_question = state["messages"][-1]
    retrieved_docs = state["retrieved_docs"]
    
    # LLM
    llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0, streaming=True)
    
    response_prompt = f"""
    User Question:
    "{user_question}"
    
    Relevant Documents:
    {retrieved_docs}
    
    Generate a concise and accurate response to the user's question based on the provided information.
    Ensure the response directly addresses the question without adding unnecessary details. Focus on clarity and accuracy when interpreting the information and constructing the reply.

    # Steps

    1. Read and understand the user's question to identify the key components and what needs to be addressed.
    2. Review the provided information carefully to ensure no important details are overlooked.
    3. Interpret the information in the context of the question.
    4. Construct a clear and concise response that directly answers the user's question, ensuring all necessary aspects are covered.

    # Output Format

    The response should be a short paragraph or bullet points, depending on the complexity and nature of the question. Aim for brevity and clarity.
    """
    
    
    # LLM을 사용하여 최종 답변 생성
    generated_response = llm(response_prompt)
    state["generated_response"] = generated_response
    print(f"Generated Response: {state['generated_response']}")
    
    rag_chain = generated_response | StrOutputParser()
    
    response = rag_chain.invoke({"context": retrieved_docs, "question": user_question})
    return {"messages": [response]}


In [17]:
workflow = StateGraph(state)

In [ ]:
workflow.add_node("agent", agent)
workflow.add_node("router", Router)
workflow.add_node("retriever_tool", retriever_tool)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("rewrite", rewrite)
workflow.add_node("generate", generate)

In [ ]:
workflow.add_edge(START, "router")
workflow.add_conditional_edges(
    "router",
    conditional_retriever,
    path_map={
        "retriever_tool": "retriever_tool",
        "agent": "agent",
    },
)
workflow.add_edge("agent", END)
workflow.add_edge("retriever_tool", "grade_documents")
workflow.add_conditional_edges(
    "grade_documents",
    # Assess agent decision
    conditional_decision,
    path_map={
        "generate": "generate",
        "rewrite": "rewrite",
    },
)
workflow.add_edge("generate", END)
workflow.add_edge("rewrite", "router")

In [20]:
graph = workflow.compile(checkpointer=memory)

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph(xray=True).draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [ ]:
ddd

In [25]:
inputs = {
    "messages": [
        {"role": "user", "content": "대물 부분 보장 범위에 대해 알려줘."},
    ],
    "session_id": session_id,
    "metadata": {},  # 필요한 경우 메타데이터 추가
    "retrieved_docs": [],
    "document_grading": [],
    "insurance_grading": [],
    "rewritten_question": [],
}

In [23]:
inputs = {
    "messages": [
        ("user", "대인 부분 약관과 보장 범위에 대해 알려줘."),
    ]
}

In [23]:
# question = {"role": "user", "content": "대물 부분 보장 범위에 대해 알려줘."}

In [24]:
# state = {
#     "messages": question,
#     "session_id": session_id,
#     "metadata": {},
#     "retrieved_docs": [],
#     "document_grading": [],
#     "insurance_grading": [],
#     "rewritten_question": []  # 초기화
# }

In [ ]:
# 스트림 실행 및 출력 처리
from pprint import pprint

config = {"configurable": {"thread_id": "user_123"}}  # 필요한 설정 추가

for output in graph.stream(inputs, config=config):
    for node, result in output.items():
        print(f"\n{'-' * 10} Output from node '{node}' {'-' * 10}\n")
        if "messages" in result:
            for message in result["messages"]:
                if isinstance(message, dict) and "content" in message:
                    print(f"**Message Content**: {message['content']}\n")
        else:
            pprint(result, indent=2, width=80, depth=None)
        print("\n" + "-" * 30 + "\n")


In [ ]:
ㅇㅇㅇ

In [ ]:
# 그래프 스트림 실행
for output in graph.stream(inputs, config=config):
    # 각 노드의 출력 처리
    for key, value in output.items():
        print(f"\n{'-'*10} Output from node '{key}' {'-'*10}\n")
        
        # 메시지가 포함된 경우만 출력
        if "messages" in value:
            for message in value["messages"]:
                if hasattr(message, "content") and message.content:
                    print(f"**Message Content**:\n{message.content}\n")
        else:
            pprint.pprint(value, indent=2, width=80, depth=None)
        
        print("\n" + "-"*30 + "\n")

In [ ]:
for output in graph.stream(inputs, config=config):
    for key, value in output.items():
        pprint.pprint(f"Output from node '{key}':")
        pprint.pprint("---")
        pprint.pprint(value, indent=2, width=80, depth=None)
    pprint.pprint("\n---\n")

In [41]:
from modules.messages import _display_message_tree

In [ ]:
_display_message_tree(output)